# 10 - Image : Optimisation et Modèles Alternatifs

## Objectif de ce notebook
**Améliorer** la baseline image du notebook 09 en testant : rééquilibrage des classes (class weights), optimisation des hyperparamètres et modèles de type boosting (XGBoost, LightGBM, CatBoost).

**Prérequis** : Exécuter le notebook 09 pour disposer des features image en cache, du label encoder et du modèle baseline dans `models/`.

**Livrable** : Meilleur modèle après optimisation, sauvegardé pour le notebook 11.

---

## Plan
1. Chargement des features et du modèle baseline (notebook 09)
2. Optimisation des hyperparamètres
3. Gestion du déséquilibre (class weights)
4. Modèles avancés (XGBoost, LightGBM, CatBoost)
5. Comparaison complète et sauvegarde du meilleur modèle

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = str(os.cpu_count() or 4)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import pickle
import warnings
warnings.filterwarnings('ignore')

sys.path.append(str(Path('../').resolve()))

from src.modeling import BaselineModels, AdvancedModels
from src.optimization import create_class_weights, random_search_optimization
from src.evaluation import evaluate_model, print_classification_report, plot_confusion_matrix

from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Bibliothèques importées")

In [ ]:
ROOT = Path('../').resolve()
MODELS_DIR = ROOT / 'models'
cache = np.load(MODELS_DIR / 'image_features_cache.npz', allow_pickle=True)
X_features = cache['X']
y_labels = cache['y']

with open(MODELS_DIR / 'label_encoder_image.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
y_encoded = label_encoder.transform(y_labels)

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_features, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

baseline_files = list(MODELS_DIR.glob('image_*_baseline.pkl'))
if baseline_files:
    baseline_model = BaselineModels.load_model(baseline_files[0])
    y_pred_baseline = baseline_model.predict(X_val_split)
    metrics_baseline = evaluate_model(y_val_split, y_pred_baseline)
else:
    from sklearn.svm import LinearSVC
    baseline_model = LinearSVC(max_iter=2000, random_state=42, dual=False)
    baseline_model.fit(X_train_split, y_train_split)
    y_pred_baseline = baseline_model.predict(X_val_split)
    metrics_baseline = evaluate_model(y_val_split, y_pred_baseline)

print(f"✅ Données : train {X_train_split.shape[0]:,} | val {X_val_split.shape[0]:,}")
print(f"   Baseline F1-macro : {metrics_baseline['f1_macro']:.4f}")

## 2. Optimisation des hyperparamètres (SVM)

In [ ]:
svm_param_dist = {'C': [0.1, 0.5, 1.0, 2.0, 5.0], 'max_iter': [1000, 2000], 'dual': [False]}
svm_model = LinearSVC(random_state=42, dual=False)
svm_optimized = random_search_optimization(svm_model, X_train_split, y_train_split, svm_param_dist, n_iter=15, cv=3, n_jobs=-1, random_state=42)
y_pred_svm_opt = svm_optimized['best_model'].predict(X_val_split)
metrics_svm_opt = evaluate_model(y_val_split, y_pred_svm_opt)
print(f"SVM optimisé F1-macro : {metrics_svm_opt['f1_macro']:.4f}")

## 3. Class weights

In [ ]:
class_weights = create_class_weights(y_train_split, method='balanced')
svm_weighted = LinearSVC(C=1.0, class_weight=class_weights, random_state=42, dual=False, max_iter=2000)
svm_weighted.fit(X_train_split, y_train_split)
y_pred_weighted = svm_weighted.predict(X_val_split)
metrics_weighted = evaluate_model(y_val_split, y_pred_weighted)
print(f"SVM + class weights F1-macro : {metrics_weighted['f1_macro']:.4f}")

## 4. Modèles avancés

In [ ]:
advanced_models = AdvancedModels(random_state=42)
advanced_models.create_advanced_models(class_weights=class_weights)
trained_advanced = advanced_models.train_all_models(X_train_split, y_train_split)
print("✅ Modèles avancés entraînés")

## 5. Comparaison et sauvegarde

In [ ]:
all_results = [
    {'Model': 'Baseline', 'F1_macro': metrics_baseline['f1_macro'], 'F1_weighted': metrics_baseline['f1_weighted']},
    {'Model': 'SVM Optimized', 'F1_macro': metrics_svm_opt['f1_macro'], 'F1_weighted': metrics_svm_opt['f1_weighted']},
    {'Model': 'SVM Class Weights', 'F1_macro': metrics_weighted['f1_macro'], 'F1_weighted': metrics_weighted['f1_weighted']},
]
for name in trained_advanced:
    yp = advanced_models.predict(name, X_val_split)
    m = evaluate_model(y_val_split, yp)
    all_results.append({'Model': name, 'F1_macro': m['f1_macro'], 'F1_weighted': m['f1_weighted']})

results_df = pd.DataFrame(all_results).sort_values('F1_macro', ascending=False)
print(results_df.to_string(index=False))

best_name = results_df.iloc[0]['Model']
print(f"\n🏆 Meilleur : {best_name}")